# Step 1b — Observable posteriors (publication figure)

Optional, diagnostic. Renders the transit-observable posteriors produced in step 1 with the
pipeline's publication style ([`photoring.plotting`](photoring/plotting.py)). Add literature or
external-catalogue overlays here for your target as needed.

In [ ]:
# ── Bootstrap: make the sibling packages importable without installation ────
# The pipeline uses three packages that live in the repository, uninstalled:
#   exorings, geotrans   (repo root)      photoring   (pipeline/)
# We locate the repo root and pipeline/ robustly from the current working dir
# (Jupyter / nbconvert / papermill all run notebooks from pipeline/).
import sys, pathlib
_HERE   = pathlib.Path.cwd().resolve()
_cands  = [_HERE, *_HERE.parents]
_NB_DIR = next((c for c in _cands if (c / "photoring").is_dir()), _HERE)
_REPO   = next((c for c in _cands if (c / "exorings").is_dir()), _NB_DIR.parent)
for _p in (str(_REPO), str(_NB_DIR)):
    if _p not in sys.path:
        sys.path.insert(0, _p)
print("repo root :", _REPO)
print("pipeline  :", _NB_DIR)

In [ ]:
import numpy as np
import photoring as pr
from photoring.io import load_observables
import photoring.plotting as plot
plot.apply_style()

CASE = "kepler_51"
PLANETS = ["b", "d"]
paths = pr.CasePaths(CASE)
ttv = {pl: load_observables(paths.observables_file(pl)) for pl in PLANETS}
print({pl: len(v["delta"]) for pl, v in ttv.items()})

## Observable posteriors, per planet

In [ ]:
plot.plot_observable_posteriors(
    ttv,
    keys=["p", "delta", "T14", "T23", "aR", "rho_obs_gcc", "b", "i_orb"],
    paths=paths, run_tag=f"{CASE}_observable_posteriors",
)

## Summary statistics

In [ ]:
import numpy as np
for pl, t in ttv.items():
    print(f"\n=== Planet {pl} ===")
    for key, lbl, scale in [("delta","delta[ppm]",1e6), ("T14","T14[h]",1.0),
                            ("T23","T23[h]",1.0), ("rho_obs_gcc","rho_obs[g/cc]",1.0),
                            ("b","b",1.0), ("aR","a/R*",1.0)]:
        v = np.asarray(t[key]) * scale
        print(f"  {lbl:>16}: {np.median(v):.4f}  [-{np.median(v)-np.percentile(v,16):.4f}, +{np.percentile(v,84)-np.median(v):.4f}]")